In [21]:
import os
import warnings
import seaborn as sns
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.model_selection import LeaveOneGroupOut, ParameterGrid
from sklearn.metrics import f1_score, balanced_accuracy_score, recall_score, precision_score, confusion_matrix
from xgboost import XGBClassifier

In [22]:
# Suppress warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
seed = 69
torch.manual_seed(seed)
np.random.seed(seed)

# Define root directory
root = '.'

In [23]:
# Load dataset
(samples, siss, ohss, okss, participants) = torch.load(os.path.join(root, 'dataset/dataset-daily.pt'), weights_only = False)

In [24]:
# Convert tensors to pandas DataFrame
df = pd.DataFrame({
    "SISS": siss,
    "OHSS": ohss,
    "OKSS": okss
})

In [25]:
df.head(3)

,SISS,OHSS,OKSS
0,25.0,25.0,31.0
1,25.0,25.0,31.0
2,25.0,25.0,31.0


In [27]:
#participants

In [6]:
# Compute quartiles for discretization
siss_q1, siss_q2, siss_q3 = np.percentile(df["SISS"], [25, 50, 75])
ohss_q1, ohss_q2, ohss_q3 = np.percentile(df["OHSS"], [25, 50, 75])
okss_q1, okss_q2, okss_q3 = np.percentile(df["OKSS"], [25, 50, 75])

In [7]:
# Define quartile-based bins and labels
quartile_labels = [0, 1, 2, 3]

In [8]:
# Apply discretization
df["SISS_Category_Q"] = pd.cut(df["SISS"], bins=[df["SISS"].min(), siss_q1, siss_q2, siss_q3, df["SISS"].max()],
                               labels=quartile_labels, include_lowest=True).astype(int)
df["OHSS_Category_Q"] = pd.cut(df["OHSS"], bins=[df["OHSS"].min(), ohss_q1, ohss_q2, ohss_q3, df["OHSS"].max()],
                               labels=quartile_labels, include_lowest=True).astype(int)

df["OKSS_Category_Q"] = pd.cut(df["OKSS"], bins=[df["OKSS"].min(), okss_q1, okss_q2, okss_q3, df["OKSS"].max()],
                               labels=quartile_labels, include_lowest=True).astype(int)

In [9]:
# Extract features, target, and patient IDs for LOPO
X = pd.DataFrame(samples)
y = df["OHSS_Category_Q"]
groups = pd.Series(participants)

In [10]:
X

,0,1,2,3,4,5,6,7,8,9,...,25,26,27,28,29,30,31,32,33,34
0,4344.0,3.6031,606.3607,9.8398,17.3118,0.6134,42744.2077,87.0,65.68,58.0,...,0.0200,76.38,60.81,55.9,2.1,409.0,106.0,21.0,37.1818,0.4583
1,5389.0,3.4781,73.4037,9.8467,3.8858,0.5211,53063.6843,114.0,70.63,58.0,...,0.0000,76.00,61.00,57.0,3.0,694.0,252.0,12.0,57.8333,0.5000
2,6074.0,3.4455,492.7764,9.8378,14.4617,0.6037,59754.7939,122.0,67.58,58.0,...,0.0000,74.00,60.00,56.0,2.0,539.0,123.0,19.0,41.4615,0.5417
3,8499.0,3.6984,261.0654,9.8094,8.9815,0.4333,83370.5096,110.0,67.44,58.0,...,0.0000,74.00,59.00,54.0,3.0,415.0,106.0,13.0,34.5833,0.5000
4,5578.0,3.9855,88.4960,9.8284,3.7246,0.4084,54822.8386,107.0,69.46,53.0,...,0.0000,82.00,61.00,56.0,4.0,447.0,185.0,8.0,63.8571,0.2917
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
569,6142.0,3.6883,86.7000,9.7764,3.6218,0.4919,60046.8127,126.0,65.57,51.0,...,0.5333,70.00,59.50,53.0,3.0,486.0,131.0,17.0,34.7143,0.5833
570,9612.0,4.2044,147.3815,9.7892,3.5972,0.3852,94094.2580,122.0,65.03,50.0,...,0.1667,67.50,57.50,50.0,5.0,1522.0,916.0,14.0,152.2000,0.4167
571,12908.0,4.3611,92.2711,9.7850,4.6104,0.4196,126304.2913,118.0,66.18,53.0,...,0.2167,72.00,57.50,48.5,2.0,925.0,326.0,17.0,84.0909,0.4583
572,4707.0,4.0588,44.3836,9.8331,3.4743,0.6853,46284.2398,131.0,72.31,53.0,...,0.0000,86.00,60.00,53.0,5.0,564.0,192.0,11.0,62.6667,0.3750


In [11]:
len(participants)

574

In [12]:
# Define classifier and hyperparameter grid
#model = XGBClassifier(use_label_encoder=False, eval_metric="mlogloss")
param_grid = {"max_depth": [3, 5, 7], 
              "n_estimators": [20, 50]}

##  https://www.geeksforgeeks.org/machine-learning/xgbclassifier/
##  https://www.geeksforgeeks.org/machine-learning/
##  https://it.wikipedia.org/wiki/Discesa_del_gradiente

In [13]:
# Leave-One-Patient-Out CV
outer_logo = LeaveOneGroupOut()

In [14]:
# Initialize results storage
overall_conf_matrix = np.zeros((4, 4))  # Assuming 4 categories (0, 1, 2, 3)
performance_metrics = []

In [15]:
indices_2 = df[df["OHSS_Category_Q"] == 2].index
indices_2

Index([303, 304, 305, 306, 307, 308, 309, 310, 311, 312, 313, 314, 315, 316,
       317, 318, 319, 320, 321, 322, 323, 324, 325, 326, 327, 328, 329, 330,
       331, 332, 333, 334, 335, 336, 337, 338, 339, 340, 341, 342, 343, 344,
       345, 346, 347, 348, 349, 350, 351, 352, 353, 354, 355, 356, 357, 358,
       359, 360, 361, 362, 363, 364, 365, 366, 367, 368, 369, 370, 371, 372,
       373, 388, 389, 390, 391, 392, 393, 394, 395, 396, 397, 398, 399, 400,
       401],
      dtype='int64')

In [16]:
participants[indices]

array([6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
       6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 7, 7,
       7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7,
       7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7],
      dtype=int64)

In [17]:
len(X)

574

In [18]:
# Outer LOPO Loop
count=0
for train_idx, test_idx in outer_logo.split(X, y, groups):
    #print(train_idx) index
    count=count+1
    print(count)
    
    if count == 6 or count == 7:
        continue 
    X_train_outer, X_test = X.iloc[train_idx].to_numpy(), X.iloc[test_idx].to_numpy()
    y_train_outer, y_test = y.iloc[train_idx].to_numpy(), y.iloc[test_idx].to_numpy()
    groups_train_outer = groups.iloc[train_idx]
    print(groups_train_outer.unique())
    
    # Inner LOPO for Hyperparameter Optimization
    inner_logo = LeaveOneGroupOut()
    best_model = None
    best_score = -np.inf
    
    for inner_train_idx, inner_val_idx in inner_logo.split(X_train_outer, y_train_outer, groups_train_outer):
        X_train_inner, X_val = X_train_outer[inner_train_idx], X_train_outer[inner_val_idx]
        y_train_inner, y_val = y_train_outer[inner_train_idx], y_train_outer[inner_val_idx]
        groups_train_inner = groups_train_outer.iloc[inner_train_idx]
        print(groups_train_inner.unique())
        
        # Hyperparameter tuning
        for params in ParameterGrid(param_grid):
            params = {k: int(v) if isinstance(v, np.generic) else v for k, v in params.items()}
            model = XGBClassifier(use_label_encoder=False, eval_metric="mlogloss")
            model.set_params(**params)
            model.fit(X_train_inner, y_train_inner)
            y_val_pred = model.predict(X_val)
            score = f1_score(y_val, y_val_pred, average="macro")
            # https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html

            if score > best_score:
                best_score = score
                best_model = model
                best_params = params  # Store the best hyperparameters

    # Train best model on full outer training set
    best_model.fit(X_train_outer, y_train_outer)
    y_pred = best_model.predict(X_test)

    # Compute metrics
    f1 = f1_score(y_test, y_pred, average="macro")
    balanced_acc = balanced_accuracy_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred, average="macro")
    precision = precision_score(y_test, y_pred, average="macro")
    conf_matrix = confusion_matrix(y_test, y_pred, labels=[0, 1, 2, 3])

    # Aggregate confusion matrices
    overall_conf_matrix += conf_matrix

    # Store results
    performance_metrics.append([f1, balanced_acc, recall, precision])

1
[ 2  3  4  5  6  7  8  9 10]
[ 3  4  5  6  7  8  9 10]
[ 2  4  5  6  7  8  9 10]
[ 2  3  5  6  7  8  9 10]
[ 2  3  4  6  7  8  9 10]
[ 2  3  4  5  7  8  9 10]
[ 2  3  4  5  6  8  9 10]
[ 2  3  4  5  6  7  9 10]
[ 2  3  4  5  6  7  8 10]
[2 3 4 5 6 7 8 9]
2
[ 1  3  4  5  6  7  8  9 10]
[ 3  4  5  6  7  8  9 10]
[ 1  4  5  6  7  8  9 10]
[ 1  3  5  6  7  8  9 10]
[ 1  3  4  6  7  8  9 10]
[ 1  3  4  5  7  8  9 10]
[ 1  3  4  5  6  8  9 10]
[ 1  3  4  5  6  7  9 10]
[ 1  3  4  5  6  7  8 10]
[1 3 4 5 6 7 8 9]
3
[ 1  2  4  5  6  7  8  9 10]
[ 2  4  5  6  7  8  9 10]
[ 1  4  5  6  7  8  9 10]
[ 1  2  5  6  7  8  9 10]
[ 1  2  4  6  7  8  9 10]
[ 1  2  4  5  7  8  9 10]
[ 1  2  4  5  6  8  9 10]
[ 1  2  4  5  6  7  9 10]
[ 1  2  4  5  6  7  8 10]
[1 2 4 5 6 7 8 9]
4
[ 1  2  3  5  6  7  8  9 10]
[ 2  3  5  6  7  8  9 10]
[ 1  3  5  6  7  8  9 10]
[ 1  2  5  6  7  8  9 10]
[ 1  2  3  6  7  8  9 10]
[ 1  2  3  5  7  8  9 10]
[ 1  2  3  5  6  8  9 10]
[ 1  2  3  5  6  7  9 10]
[ 1  2  3  5  6 

In [19]:
# Convert results to DataFrame
performance_df = pd.DataFrame(performance_metrics, columns=["Macro-F1", "Balanced Accuracy", "Macro Recall", "Macro Precision"])

# Save performance metrics and overall confusion matrix
output_path = os.path.join(root, "results/model_performance_ohss.xlsx")
with pd.ExcelWriter(output_path) as writer:
    performance_df.to_excel(writer, sheet_name="All_Folds")

In [20]:
# Plot overall confusion matrix
plt.figure(figsize=(6, 5))
sns.heatmap(overall_conf_matrix, annot=True, cmap="coolwarm", xticklabels=[0, 1, 2, 3], yticklabels=[0, 1, 2, 3])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Overall Confusion Matrix OHSS")
conf_matrix_path = os.path.join(root, "results/overall_confusion_matrix_ohss.pdf")
plt.savefig(conf_matrix_path, dpi=300, bbox_inches='tight')
plt.close()

print(f"\n✅ Performance metrics saved as: {output_path}")
print(f"✅ Overall Confusion Matrix saved as: {conf_matrix_path}")


✅ Performance metrics saved as: .\results/model_performance_ohss.xlsx
✅ Overall Confusion Matrix saved as: .\results/overall_confusion_matrix_ohss.pdf
